## Load input data and trained model

In [ ]:
import anndata as ad
import crested

# Load data
adata = ad.read_h5ad("results/CREsted/nematostella_celltype_500region_250target_logcount_meansubtract.h5ad")

# Load genome
genome_file = "genome/Nvec_vc1.1_gDNA.fasta"
chrom_sizes = "genome/Nvec_vc1.1_gDNA.fasta.fai"
genome = crested.Genome(genome_file, chrom_sizes)

# Register the genome so that it can be used by the package
crested.register_genome(genome)

2025-03-01T11:04:47.393650+0100 INFO Genome Nvec_vc1.1_gDNA registered.


Load a trained model

In [ ]:
import keras

# Load model
model_path = "results/CREsted/models/meansubstract_deeptopic_celltype_500_logcount_huber_finetuned.keras"
model = keras.models.load_model(model_path, compile=False)

In [3]:
# Store predictions for all our regions in the anndata object for later inspection.
predictions = crested.tl.predict(adata, model)
adata.layers["model"] = predictions.T  # adata expects (C, N) instead of (N, C)

  20/3523 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step   

/users/asebe/aelek/bin/miniconda3/lib/python3.12/site-packages/keras/src/backend/torch/nn.py:466: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  outputs = tnn.conv1d(


3523/3523 ━━━━━━━━━━━━━━━━━━━━ 23s 6ms/step


## Contributions in top selected cell type regions

To obtain cell type-characteristic patterns, we need to calculate contribution scores on highly specific regions

### Filter correlated regions

We can first select the peaks for which the model predictions correlate well with the ground truth peak heights.

In [ ]:
import numpy as np
import anndata as an
from scipy.stats import pearsonr

# Copy the peak heights
adata_filtered = adata.copy()

# Correlation of peak heights and model predictions
pcc = np.array([pearsonr(adata_filtered.X[:,i], adata_filtered.layers["model"][:,i])[0] for i in range(adata_filtered.X.shape[1])])
adata_filtered.var["PCC"] = pcc
pcc_mask = pcc > 0.5

# Filter peaks data
regions_filtered = adata_filtered.var_names[pcc_mask]
adata_filtered._inplace_subset_var(regions_filtered)
adata_filtered

In [ ]:
%matplotlib inline

import os
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
from scipy.stats import pearsonr, spearmanr

def reg_class(adata, split="test", predictions="model_500_ft", metric="pearson"):
    # Get indices of test regions
    split_mask = adata.var["split"] == split

    # Subset predictions and true values
    y_preds = adata.layers[predictions][:, split_mask]
    y_true = adata.X[:, split_mask]

    # Convert sparse matrix to dense if necessary
    if hasattr(y_true, "toarray"):
        y_true = y_true.toarray()

    if metric == "pearson":
        return np.array([pearsonr(y_true[i], y_preds[i])[0] for i in range(y_true.shape[0])])

    elif metric == "spearman":
        return np.array([spearmanr(y_true[i], y_preds[i])[0] for i in range(y_true.shape[0])])

    elif metric == "rmse":
        return np.sqrt(np.mean((y_preds - y_true) ** 2, axis=1))

    elif metric == "mae":
        return np.mean(np.abs(y_preds - y_true), axis=1)

    elif metric == "mape":
        return np.mean(np.abs((y_true - y_preds) / y_true), axis=1) * 100

    elif metric == "r2":
        ss_total = np.sum((y_true - np.mean(y_true, axis=1, keepdims=True))**2, axis=1)
        ss_residual = np.sum((y_true - y_preds)**2, axis=1)
        return 1 - (ss_residual / ss_total)

    elif metric == "explained_variance":
        return 1 - (np.var(y_true - y_preds, axis=1) / np.var(y_true, axis=1))

    elif metric == "msle":
        return np.mean((np.log1p(y_true) - np.log1p(y_preds))**2, axis=1)

    else:
        raise ValueError(f"Unknown metric: {metric}")

base_model_pcc = reg_class(adata_filtered, split="test", predictions="model", metric="pearson")

# Get sample names (assuming they represent different classes)
sample_names = list(adata_filtered.obs_names)
x_indices = np.arange(len(sample_names))  # Numeric x-axis positions

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 4))

# Scatter plot for both models
ax.bar(x_indices, base_model_pcc)

# Customize plot
ax.set_ylabel("Test PCC")
ax.set_title("PCC per Class")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_xticks(x_indices)
ax.set_xticklabels(sample_names, rotation=45, ha="right")

plt.tight_layout()
# plt.savefig(os.path.join(plot_dir, "prediction-test-PCC.pdf"), format="pdf", bbox_inches="tight")


### Select the most informative regions per cell type

There are different options to select the top regios per cell type: either purely based on peak height, purely based on predictions, or on their combination. 
Here we select top regions based on combination of peak hight and model predictions.

In [4]:
import anndata as an

# Copy the input data (original or filtered for correlated peaks)
adata_combined = adata.copy()

# Take the average with the predictions
adata_combined.X = (
    adata_combined.X + adata_combined.layers["model"]
) / 2

# Take the most specific regions
top_k = 2000
crested.pp.sort_and_filter_regions_on_specificity(
    adata_combined, top_k=top_k, method="gini"
)

adata_combined

2025-03-01T11:06:14.761793+0100 INFO After sorting and filtering, kept 44000 regions.


/users/asebe/aelek/bin/miniconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 22 × 44000
    obs: 'file_path'
    var: 'chr', 'start', 'end', 'split', 'Class name', 'rank', 'gini_score'
    layers: 'model'

In [ ]:
import os

# Where to save results
modisco_dir = "results/CREsted/modisco/meansubstract_deeptopic_celltype_500_logcount_huber_finetuned/gini_2000"
os.makedirs(modisco_dir, exist_ok=True)

# Save input dat
adata_combined.write_h5ad(os.path.join(modisco_dir, "adata.h5ad"))

### Inspect selected regions

We will plot imprtance scores for top regions in retractor muscle cell types.

In [ ]:
import os
import anndata as ad

# Where we saved data
modisco_dir = "results/CREsted/modisco/meansubstract_deeptopic_celltype_500_logcount_huber_finetuned/pearsonr_gini_1000"

# Load data
adata = ad.read_h5ad(os.path.join(modisco_dir, "adata.h5ad"))

# Cell types we want to show
classes_of_interest = ["muscle_tentacle_retractor", "muscle_mesentery_retractor"]
class_idx = list(adata.obs_names.get_indexer(classes_of_interest))

cell_type_mask = adata.var["Class name"].isin(classes_of_interest)
filtered_var = adata.var[cell_type_mask]

# Get first 20 regions per class
regions_of_interest = (
    filtered_var.groupby("Class name", observed=False)
    .head(100)
    .index
)
filtered_var = filtered_var.loc[regions_of_interest]

/users/asebe/aelek/bin/miniconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [39]:
filtered_var

,chr,start,end,split,PCC,Class name,rank,gini_score
region,,,,,,,,
NC_064046.1:13001928-13002428,NC_064046.1,13001928,13002428,train,0.734835,muscle_mesentery_retractor,1,0.829812
NC_064045.1:4326184-4326684,NC_064045.1,4326184,4326684,train,0.673377,muscle_mesentery_retractor,2,0.824805
NC_064036.1:14388543-14389043,NC_064036.1,14388543,14389043,train,0.587932,muscle_mesentery_retractor,3,0.812137
NC_064041.1:15531691-15532191,NC_064041.1,15531691,15532191,train,0.606718,muscle_mesentery_retractor,4,0.803321
NC_064042.1:6756483-6756983,NC_064042.1,6756483,6756983,train,0.604747,muscle_mesentery_retractor,5,0.802755
...,...,...,...,...,...,...,...,...
NC_064034.1:9640586-9641086,NC_064034.1,9640586,9641086,val,0.559429,muscle_tentacle_retractor,96,0.666672
NC_064043.1:11774327-11774827,NC_064043.1,11774327,11774827,train,0.651277,muscle_tentacle_retractor,97,0.665798
NC_064043.1:1913756-1914256,NC_064043.1,1913756,1914256,train,0.545271,muscle_tentacle_retractor,98,0.665461


Now we select peaks assigned to a set of genes we are interested in.

In [ ]:
import pandas as pd

# Load region-to-gene assignment and preprocess
asgn = pd.read_csv("results/CREsted/example_muscle_genes.csv")
asgn["mid"] = (asgn["start"] + asgn["end"]) / 2
asgn.rename(columns={asgn.columns[0]: "chr"}, inplace=True)

# Compute midpoints for model regions
filtered_var["mid"] = (filtered_var["start"] + filtered_var["end"]) / 2

# Ensure chromosome names match types
asgn["chr"] = asgn["chr"].astype(str)
filtered_var["chr"] = filtered_var["chr"].astype(str)

# Merge and find overlaps
merged = filtered_var.merge(asgn, on="chr", suffixes=("_model", "_gasn"))
overlapping_regions = merged[
    (merged["start_model"] <= merged["end_gasn"]) & 
    (merged["end_model"] >= merged["start_gasn"])
].copy()

# Create region identifiers
overlapping_regions["region_id"] = overlapping_regions.apply(
    lambda row: f"{row['chr']}:{row['start_model']}-{row['end_model']}", axis=1
)
overlapping_regions = overlapping_regions[["region_id", "Class name", "rank", "gini_score", "split", "PCC"]]
overlapping_regions = overlapping_regions.drop_duplicates()
overlapping_regions.set_index("region_id", inplace=True)

In [41]:
overlapping_regions

,Class name,rank,gini_score,split,PCC
region_id,,,,,
NC_064036.1:8537525-8538025,muscle_mesentery_retractor,10,0.788607,train,0.652048
NC_064034.1:16065045-16065545,muscle_mesentery_retractor,29,0.770788,val,0.778722
NC_064036.1:5763949-5764449,muscle_mesentery_retractor,45,0.762913,train,0.626587
NC_064041.1:9459745-9460245,muscle_mesentery_retractor,57,0.759908,train,0.525241
NC_064037.1:9835069-9835569,muscle_mesentery_retractor,62,0.759196,train,0.562783
NC_064047.1:926348-926848,muscle_mesentery_retractor,80,0.756396,train,0.576493
NC_064042.1:10040168-10040668,muscle_mesentery_retractor,89,0.753259,train,0.576401
NC_064047.1:2141969-2142469,muscle_tentacle_retractor,5,0.749042,train,0.512975
NC_064036.1:3528257-3528757,muscle_tentacle_retractor,16,0.720894,train,0.524429


In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages

# WHere to save the plots
plot_dir = os.path.join("plots", "CREsted", "muscle_contrib")
os.makedirs(plot_dir, exist_ok=True)

for region, row in overlapping_regions.iterrows():

    cell_type = row["Class name"] 
    split = row["split"]
    rank = row["rank"]
    pcc = row["PCC"]
    
    # Plot prediction vs ground truth for all cell types
    crested.pl.bar.region_predictions(
        adata, region,
        share_y=False, 
        x_label_rotation=90,
        title=f"{cell_type} {rank}: {region} PCC={pcc}",
        width = 12, height = 6,
        save_path=os.path.join(plot_dir, f"{cell_type}-{rank}-{region}-prediction.pdf")
    )

    # Calculate importance scores for selected cell types
    scores, one_hot_encoded_sequence = crested.tl.contribution_scores(
        region,
        target_idx=class_idx,
        model=model,
    )

    # Plot importance scores
    fig = crested.pl.patterns.contribution_scores(
        scores,
        one_hot_encoded_sequence,
        class_labels=classes_of_interest,
        zoom_n_bases=250,
        width = 15, height = 3,
        save_path=os.path.join(plot_dir, f"{cell_type}-{rank}-contribution.pdf")
    )


### Calculate contribution scores per cell type

Next we calculate contribution scores in selected sequences for each class. By default, the contribution scores are calculated using the expected integrated gradients method.

In [6]:
# Cell type idx
for ct in adata_combined.obs_names:
    print(f"{ct}: {list(adata_combined.obs_names).index(ct)}")

cnidocyte: 0
digestive_filaments_1: 1
digestive_filaments_2: 2
digestive_filaments_3: 3
epidermis_1: 4
epidermis_2: 5
gastro_IRF1_2: 6
gastro_circular_muscle_1: 7
gastro_circular_muscle_2: 8
gastro_parietal_muscle: 9
gastro_somatic_gonad: 10
gland: 11
muscle_mesentery_retractor: 12
muscle_tentacle_retractor: 13
neuron_GATA_Islet_1: 14
neuron_GATA_Islet_2: 15
neuron_Pou4_FoxL2_1: 16
neuron_Pou4_FoxL2_2: 17
neuron_Pou4_FoxL2_3: 18
precursors_NPC: 19
precursors_PGC: 20
precursors_endoNPC: 21


In [7]:
import numpy as np

# Calculate contributions
crested.tl.contribution_scores_specific(
    input=adata_combined,
    target_idx=None,
    model=model,
    output_dir=modisco_dir,
)

2025-03-01T11:06:31.337032+0100 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


Model: 100%|██████████| 1/1 [14:46<00:00, 886.62s/it]


2025-03-01T11:21:22.837546+0100 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


Model: 100%|██████████| 1/1 [14:45<00:00, 885.69s/it]


2025-03-01T11:36:13.341644+0100 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


...

2025-03-01T12:20:46.220467+0100 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


Model: 100%|██████████| 1/1 [14:46<00:00, 886.29s/it]


2025-03-01T12:35:37.355773+0100 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


Model: 100%|██████████| 1/1 [14:45<00:00, 885.94s/it]


(array([[[[-2.46529758e-04,  8.82471271e-04,  3.43473628e-04, ...,
           -6.62143517e-04, -4.64601413e-04, -3.11964104e-04],
          [ 5.86824666e-04, -1.40751409e-03, -9.50180576e-04, ...,
           -1.02782709e-04,  5.71037701e-04,  7.35404261e-04],
          [-1.02954183e-03,  1.29504665e-03, -1.15851195e-04, ...,
            2.88449257e-04, -1.17794948e-03, -7.43529352e-04],
          [ 4.39023162e-04, -1.08283269e-03,  4.63585136e-04, ...,
           -2.12202052e-04,  3.79204517e-04, -3.86078755e-04]]],
 
 
        [[[ 2.02581956e-04, -1.42682681e-03, -3.61291645e-03, ...,
            2.79071508e-04,  3.57725767e-05,  4.48220904e-04],
          [ 7.74770684e-04, -1.21703336e-03,  4.38956777e-03, ...,
            3.88798449e-04, -1.25338370e-03, -7.07977379e-05],
          [-9.95358569e-04,  1.62191002e-03, -4.17728815e-03, ...,
           -9.57471435e-04,  3.40210507e-04, -8.72633187e-04],
          [-9.04187793e-04, -1.27254069e-04,  1.97205320e-03, ...,
           -6.683

## Running tfmodisco-lite

Run TFModisco-lite on the saved contribution scores to find motifs that are important for the classification/regression task.

In [ ]:
# meme_db, motif_to_tf_file = crested.get_motif_db()

# Run tfmodisco on the contribution scores
crested.tl.modisco.tfmodisco(
    window=250,
    output_dir=modisco_dir,
    contrib_dir=modisco_dir,
    report=True,
    meme_db="results/CREsted/motif-archetypes-PPM-PCCnorm-0.8-IC0.5-8bp-unique-pwms.meme",
    max_seqlets=10000,
)

The code above gives an error. We will run TF Modisco from command line instead.

```
conda activate tfmodisco_lite

DIR=/users/asebe/aelek/proj/nvec_crested/
MODEL=meansubstract_deeptopic_celltype_500_logcount_huber_finetuned
OUT_DIR=${DIR}/modisco/${MODEL}/gini_2000
mkdir -p ${OUT_DIR}

CELL_TYPES="cnidocyte digestive_filaments_1 digestive_filaments_2 digestive_filaments_3 epidermis_1 epidermis_2 gastro_circular_muscle_1 gastro_circular_muscle_2 gastro_IRF1_2 gastro_parietal_muscle gastro_somatic_gonad gland muscle_mesentery_retractor muscle_tentacle_retractor neuron_GATA_Islet_1 neuron_GATA_Islet_2 neuron_Pou4_FoxL2_1 neuron_Pou4_FoxL2_2 neuron_Pou4_FoxL2_3 precursors_endoNPC precursors_NPC precursors_PGC"

for NAME in ${CELL_TYPES}
do
echo $NAME
ls -l ${OUT_DIR}/${NAME}_oh.npz 
ls -l ${OUT_DIR}/${NAME}_contrib.npz 
done

for NAME in ${CELL_TYPES}
do
sbatch ${DIR}/scripts/tfmodisco.sh ${NAME} ${OUT_DIR} \
    -s ${OUT_DIR}/${NAME}_oh.npz \
    -a ${OUT_DIR}/${NAME}_contrib.npz \
    -w 250 -n 10000
done
```